In [8]:
import pandas as pd
import math
import numpy as np

df = pd.read_csv('christmas_movies.csv')

In [9]:
print(df.columns.tolist())

['title', 'rating', 'runtime', 'imdb_rating', 'meta_score', 'genre', 'release_year', 'description', 'director', 'stars', 'votes', 'gross', 'img_src', 'type']


In [10]:
print(df.isnull().sum())

title             0
rating          213
runtime          41
imdb_rating      34
meta_score      773
genre             1
release_year     11
description       0
director          5
stars            11
votes            34
gross           794
img_src           0
type              0
dtype: int64


In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 873 entries, 0 to 872
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         873 non-null    object 
 1   rating        660 non-null    object 
 2   runtime       832 non-null    float64
 3   imdb_rating   839 non-null    float64
 4   meta_score    100 non-null    float64
 5   genre         872 non-null    object 
 6   release_year  862 non-null    float64
 7   description   873 non-null    object 
 8   director      868 non-null    object 
 9   stars         862 non-null    object 
 10  votes         839 non-null    object 
 11  gross         79 non-null     object 
 12  img_src       873 non-null    object 
 13  type          873 non-null    object 
dtypes: float64(4), object(10)
memory usage: 95.6+ KB
None


In [12]:
def parse_items(cell):
    if pd.isna(cell):
        return []

    items = [item.strip() for item in cell.split(',')]

    items = [item for item in items if item]

    return items

In [13]:
df['stars_list'] = df['stars'].apply(parse_items)
df['genre_list'] = df['genre'].apply(parse_items)

In [14]:
df.to_csv('movies_processed.csv', index=False)

In [15]:
df2 = pd.read_csv('test/movies_processed.csv')

In [16]:
df2.columns.tolist()

['title',
 'rating',
 'runtime',
 'imdb_rating',
 'meta_score',
 'genre',
 'release_year',
 'description',
 'director',
 'stars',
 'votes',
 'gross',
 'img_src',
 'type',
 'stars_list',
 'genre_list']

In [17]:
df_clean = df.dropna(subset=['imdb_rating', 'runtime']).copy()
df_clean.reset_index(drop=True, inplace=True)

In [18]:
print(df_clean.shape)

(822, 16)


In [19]:
df_clean.columns.tolist()

['title',
 'rating',
 'runtime',
 'imdb_rating',
 'meta_score',
 'genre',
 'release_year',
 'description',
 'director',
 'stars',
 'votes',
 'gross',
 'img_src',
 'type',
 'stars_list',
 'genre_list']

In [9]:
min_rtg = df_clean['imdb_rating'].min()
max_rtg = df_clean['imdb_rating'].max()
df_clean['rating_norm'] = (df_clean['imdb_rating'] - min_rtg) / (max_rtg - min_rtg)

In [21]:
min_runtime = df_clean['runtime'].min()
max_runtime = df_clean['runtime'].max()
df_clean['runtime_norm'] = (df_clean['runtime'] - min_runtime) / (max_runtime - min_runtime)

In [22]:
df_clean.isnull().sum()

title             0
rating          168
runtime           0
imdb_rating       0
meta_score      722
genre             0
release_year      0
description       0
director          0
stars             1
votes             0
gross           743
img_src           0
type              0
stars_list        0
genre_list        0
runtime_norm      0
dtype: int64

In [23]:
def jaccard_similarity(list1, list2):
    set1 = set(list1) if isinstance(list1, list) else set()
    set2 = set(list2) if isinstance(list2, list) else set()
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union != 0 else 0.0

In [24]:
print(df_clean.head())

                                   title rating  runtime  imdb_rating  \
0                          Love Actually      R    135.0          7.6   
1                             Home Alone     PG    103.0          7.7   
2  National Lampoon's Christmas Vacation  PG-13     97.0          7.5   
3                                    Elf     PG     97.0          7.1   
4         How the Grinch Stole Christmas     PG    104.0          6.3   

   meta_score                      genre  release_year  \
0        55.0     Comedy, Drama, Romance        2003.0   
1        63.0             Comedy, Family        1990.0   
2        49.0                     Comedy        1989.0   
3        66.0  Adventure, Comedy, Family        2003.0   
4        46.0    Comedy, Family, Fantasy        2000.0   

                                         description             director  \
0  Follows the lives of eight very different coup...       Richard Curtis   
1  An eight-year-old troublemaker, mistakenly lef...      

In [25]:
def euclidean_distance(row1, row2):
    return math.sqrt(
        (row1['rating_norm'] - row2['rating_norm'])**2 +
        (row1['runtime_norm'] - row2['runtime_norm'])**2
    )

In [26]:
MAX_DIST = math.sqrt(2)

In [27]:
def numeric_similarity(row1, row2):
    dist = euclidean_distance(row1, row2)

    return 1 - (dist / MAX_DIST)

In [28]:
def combined_similarity(row1, row2, w_numeric=0.4, w_genre=0.3, w_stars=0.3):
    """
    Взвешенная сумма сходств:
    - числовое (нормализованные рейтинг и длительность)
    - жанровое (Жаккард)
    - актёрское (Жаккард)
    """
    sim_num = numeric_similarity(row1, row2)
    sim_gen = jaccard_similarity(row1['genre_list'], row2['genre_list'])
    sim_strs = jaccard_similarity(row1['actors_list'], row2['actors_list'])
    return w_numeric * sim_num + w_genre * sim_gen + w_stars * sim_strs

In [29]:
indices = df_clean.index.tolist()

print("Фильмы для сравнения:")
for i, idx in enumerate(indices):
    row = df_clean.loc[idx]
    print(f"{i+1}. {row['title']} (рейтинг = {row['imdb_rating']}, длит. = {int(row['runtime'])} мин, "
          f"жанры = {str(row['genre_list'])[1:-1]}, актёров = {len(row['stars_list'])})")
print("\n")

Фильмы для сравнения:
1. Love Actually (рейтинг = 7.6, длит. = 135 мин, жанры = 'Comedy', 'Drama', 'Romance', актёров = 4)
2. Home Alone (рейтинг = 7.7, длит. = 103 мин, жанры = 'Comedy', 'Family', актёров = 4)
3. National Lampoon's Christmas Vacation (рейтинг = 7.5, длит. = 97 мин, жанры = 'Comedy', актёров = 4)
4. Elf (рейтинг = 7.1, длит. = 97 мин, жанры = 'Adventure', 'Comedy', 'Family', актёров = 4)
5. How the Grinch Stole Christmas (рейтинг = 6.3, длит. = 104 мин, жанры = 'Comedy', 'Family', 'Fantasy', актёров = 4)
6. The Grinch (рейтинг = 6.4, длит. = 85 мин, жанры = 'Animation', 'Comedy', 'Family', актёров = 5)
7. Die Hard (рейтинг = 8.2, длит. = 132 мин, жанры = 'Action', 'Thriller', актёров = 4)
8. Home Alone 2: Lost in New York (рейтинг = 6.9, длит. = 120 мин, жанры = 'Adventure', 'Comedy', 'Crime', актёров = 4)
9. The Polar Express (рейтинг = 6.6, длит. = 100 мин, жанры = 'Animation', 'Adventure', 'Comedy', актёров = 4)
10. It's a Wonderful Life (рейтинг = 8.6, длит. = 130 

In [30]:
df.to_csv('movies_processed2.csv', index=False)

In [31]:
df_clean.to_csv('movies_processed3.csv', index=False)

In [53]:
print(df_clean.head())

                                   title rating  runtime  imdb_rating  \
0                          Love Actually      R    135.0          7.6   
1                             Home Alone     PG    103.0          7.7   
2  National Lampoon's Christmas Vacation  PG-13     97.0          7.5   
3                                    Elf     PG     97.0          7.1   
4         How the Grinch Stole Christmas     PG    104.0          6.3   

   meta_score                      genre  release_year  \
0        55.0     Comedy, Drama, Romance        2003.0   
1        63.0             Comedy, Family        1990.0   
2        49.0                     Comedy        1989.0   
3        66.0  Adventure, Comedy, Family        2003.0   
4        46.0    Comedy, Family, Fantasy        2000.0   

                                         description             director  \
0  Follows the lives of eight very different coup...       Richard Curtis   
1  An eight-year-old troublemaker, mistakenly lef...      